In [1]:
import glob, pandas as pd, xarray as xr

# ---------- config ----------
DATA_ROOT = "/glade/campaign/collections/gdex/data/d633000/e5.oper.an.sfc"
DEFAULT_CHUNKS = {"time": 120, "latitude": -1, "longitude": -1}

# Patterns: each has one '{}' for YYYYMM
PATTERNS = {
    "MSL":  f"{DATA_ROOT}" + "/{}/e5.oper.an.sfc.128_151_msl.ll025sc.*.nc",
    "TCWV": f"{DATA_ROOT}" + "/{}/e5.oper.an.sfc.128_137_tcwv.ll025sc.*.nc",
    "VAR_10U":  f"{DATA_ROOT}" + "/{}/e5.oper.an.sfc.128_165_10u.ll025sc.*.nc",
    "VAR_10V":  f"{DATA_ROOT}" + "/{}/e5.oper.an.sfc.128_166_10v.ll025sc.*.nc",
}

# ---------- helpers ----------
def _months(start, end):
    s, e = pd.to_datetime(start), pd.to_datetime(end)
    return [p.strftime("%Y%m") for p in pd.period_range(s, e, freq="M")]

def _files(pattern, start, end):
    fs = []
    for ym in _months(start, end):
        fs += glob.glob(pattern.format(ym))  # fill {YYYYMM}
    fs = sorted(fs)
    if not fs: raise FileNotFoundError(pattern)
    return fs

def _preproc(lat_min=None, lat_max=None, lon_min=None, lon_max=None):
    def _pp(ds):
        # ERA5 latitude is descending (90→-90)
        if (lat_min is not None) and (lat_max is not None):
            ds = ds.sel(latitude=slice(lat_max, lat_min))
        # ERA5 longitude is native 0..360 (no wrapping)
        if (lon_min is not None) and (lon_max is not None):
            ds = ds.sel(longitude=slice(lon_min, lon_max))
        return ds
    return _pp

def _open_var(name, pattern, start, end, *, lat_min=None, lat_max=None, lon_min=None, lon_max=None):
    ds = xr.open_mfdataset(
        _files(pattern, start, end),
        combine="by_coords", engine="h5netcdf", parallel=True,
        chunks=DEFAULT_CHUNKS,
        preprocess=_preproc(lat_min, lat_max, lon_min, lon_max),
        data_vars="minimal", coords="minimal", compat="override", join="override",
    ).sortby("time").sel(time=slice(pd.to_datetime(start), pd.to_datetime(end)))
    var = list(ds.data_vars)[0] if len(ds.data_vars)==1 else name
    if var not in ds.data_vars:
        raise ValueError(f"{name}: available {list(ds.data_vars)}")
    return ds[[var]].rename({var: name})

def open_four(start, end, *, lat_min=None, lat_max=None, lon_min=None, lon_max=None):
    parts = []
    for name, pat in PATTERNS.items():
        parts.append(_open_var(name, pat, start, end,
                               lat_min=lat_min, lat_max=lat_max,
                               lon_min=lon_min, lon_max=lon_max))
    return xr.merge(parts, compat="override", join="override")
     # msl, tcwv, u10, v10 over Jun–Nov 2017


In [2]:
# ---------- example: full 2017 hurricane season ----------
# if __name__ == "__main__":
    # Atlantic box in native 0..360 longitudes (≈ -110..-10 in -180..180)
LAT_MIN, LAT_MAX = 15.0, 35.0
LON_MIN, LON_MAX = 260.0, 330.0 

ds = open_four("2017-06-01", "2017-12-01",
               lat_min=LAT_MIN, lat_max=LAT_MAX,
               lon_min=LON_MIN, lon_max=LON_MAX) 

/glade/work/ncheruku/conda-envs/era5_expv1/lib/python3.13/site-packages/dask/_task_spec.py:758: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 120. This could degrade performance. Instead, consider rechunking after loading.
  return self.func(*new_argspec, **kwargs)
/glade/work/ncheruku/conda-envs/era5_expv1/lib/python3.13/site-packages/dask/_task_spec.py:758: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 120. This could degrade performance. Instead, consider rechunking after loading.
  return self.func(*new_argspec, **kwargs)
/glade/work/ncheruku/conda-envs/era5_expv1/lib/python3.13/site-packages/dask/_task_spec.py:758: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 120. This could degrade performance. Instead, consider rechunking after loading.
  return self.func(*new_argspec, **kwargs)
/glade/work/ncheruku/conda-envs/er

In [3]:
ds

<xarray.Dataset> Size: 2GB
Dimensions:    (time: 4393, latitude: 81, longitude: 281)
Coordinates:
  * latitude   (latitude) float64 648B 35.0 34.75 34.5 34.25 ... 15.5 15.25 15.0
  * longitude  (longitude) float64 2kB 260.0 260.2 260.5 ... 329.5 329.8 330.0
  * time       (time) datetime64[ns] 35kB 2017-06-01 ... 2017-12-01
Data variables:
    MSL        (time, latitude, longitude) float32 400MB dask.array<chunksize=(120, 81, 281), meta=np.ndarray>
    TCWV       (time, latitude, longitude) float32 400MB dask.array<chunksize=(120, 81, 281), meta=np.ndarray>
    VAR_10U    (time, latitude, longitude) float32 400MB dask.array<chunksize=(120, 81, 281), meta=np.ndarray>
    VAR_10V    (time, latitude, longitude) float32 400MB dask.array<chunksize=(120, 81, 281), meta=np.ndarray>
Attributes:
    DATA_SOURCE:          ECMWF: https://cds.climate.copernicus.eu, Copernicu...
    NETCDF_CONVERSION:    CISL RDA: Conversion from ECMWF GRIB1 data to netCDF4.
    NETCDF_VERSION:       4.6.1
    CONVERSION_PLATFORM:  Linux casper20 3.10.0-693.21.1.el7.x86_64 #1 SMP We...
    CONVERSION_DATE:      Thu Sep  5 11:06:33 MDT 2019
    Conventions:          CF-1.6
    NETCDF_COMPRESSION:   NCO: Precision-preserving compression to netCDF4/HD...
    history:              Thu Sep  5 11:06:53 2019: ncks -4 --ppc default=7 e...
    NCO:                  netCDF Operators version 4.7.4 (http://nco.sf.net)

In [4]:
# def show_rgb_and_channels(rgb_da, parts=None):
#     """
#     Show the combined RGB plus the three individual channels in a 2x2 grid.
#     parts can be the dict returned by storms_rgb_for_day for labeled channels.
#     """
#     arr = rgb_da.values  # (lat, lon, 3) in [0,1]

#     fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)

#     # Flatten for easier iteration
#     axes = axes.ravel()

#     # --- RGB composite ---
#     axes[0].imshow(arr, origin="upper",
#                    extent=[float(rgb_da.longitude.min()), float(rgb_da.longitude.max()),
#                            float(rgb_da.latitude.min()),  float(rgb_da.latitude.max())],
#                    aspect="equal")
#     axes[0].set_title("RGB composite")

#     # --- individual channels ---
#     cmap_list = ["Reds", "Greens", "Blues"]
#     titles = ["R: -MSLP anomaly", "G: max ws10", "B: TCWV"]

#     for i in range(3):
#         axes[i+1].imshow(arr[:, :, i], origin="upper", cmap=cmap_list[i],
#                          extent=[float(rgb_da.longitude.min()), float(rgb_da.longitude.max()),
#                                  float(rgb_da.latitude.min()),  float(rgb_da.latitude.max())],
#                          aspect="equal")
#         axes[i+1].set_title(titles[i])

#     for ax in axes:
#         ax.set_xlabel("Longitude (0–360)")
#         ax.set_ylabel("Latitude")

#     plt.show()


In [5]:
def show_rgb_and_channels(rgb_da, parts=None):
    """
    Show the combined RGB plus the three individual channels in a 2x2 grid.
    Each subplot is square, and the image is stretched to fill (no empty space).
    """
    arr = rgb_da.values  # (lat, lon, 3) in [0,1]

    fig, axes = plt.subplots(2, 2, figsize=(10, 10), constrained_layout=True)
    axes = axes.ravel()

    # --- RGB composite ---
    axes[0].imshow(arr, origin="upper",
                   extent=[float(rgb_da.longitude.min()), float(rgb_da.longitude.max()),
                           float(rgb_da.latitude.min()),  float(rgb_da.latitude.max())],
                   aspect="auto")   # stretch to fit square
    axes[0].set_title("RGB composite")

    # --- individual channels ---
    cmap_list = ["Reds", "Greens", "Blues"]
    titles = ["R: -MSLP anomaly", "G: max ws10", "B: TCWV"]

    for i in range(3):
        axes[i+1].imshow(arr[:, :, i], origin="upper", cmap=cmap_list[i],
                         extent=[float(rgb_da.longitude.min()), float(rgb_da.longitude.max()),
                                 float(rgb_da.latitude.min()),  float(rgb_da.latitude.max())],
                         aspect="auto")   # stretch to fit square
        axes[i+1].set_title(titles[i])

    for ax in axes:
        ax.set_box_aspect(1)  # make subplot square
        ax.set_xlabel("Longitude (0–360)")
        ax.set_ylabel("Latitude")

    plt.show()


In [6]:
# rgb, parts = storms_rgb_for_day(ds, mean_msl, date="2017-09-21",
#                                 r_range=(-20,20), g_range=(0,35), b_range=(20,70),
#                                 tcwv_nonlin=None)

# show_rgb_and_channels(rgb, parts)


In [7]:
import numpy as np
import xarray as xr
import pandas as pd
from pathlib import Path
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

# ---------------- config ----------------
# Fixed ranges (good hurricane defaults)
R_RANGE = (-20.0, 20.0)   # MSLP anomaly [hPa]
G_RANGE = (0.0, 35.0)     # 10m wind daily max [m/s]
B_RANGE = (20.0, 70.0)    # TCWV daily mean [kg m^-2]; set to (10,60) if mid-lats heavy
TCWV_NONLIN = "sqrt"      # None | "sqrt" | "log"
OUT_ROOT = Path("./openclip_ready_512"); OUT_ROOT.mkdir(exist_ok=True)
(OUT_ROOT/"red").mkdir(exist_ok=True); (OUT_ROOT/"green").mkdir(exist_ok=True)
(OUT_ROOT/"blue").mkdir(exist_ok=True); (OUT_ROOT/"rgb").mkdir(exist_ok=True)

SAVE_SIZE = 512
JPEG_QUALITY = 90
N_WORKERS = 8  # threads for writing JPEGs (I/O bound)

# ---------------- helpers ----------------
def _get(ds, names):
    for n in names:
        if n in ds: return ds[n]
    raise KeyError(f"Vars {list(ds.data_vars)} do not include any of {names}")

def _rescale01(da, vmin, vmax, eps=1e-6, nonlin=None):
    x = (da - vmin) / (vmax - vmin + eps)
    x = x.clip(0, 1)
    if nonlin == "sqrt":
        x = xr.apply_ufunc(np.sqrt, x, dask="allowed")
    elif nonlin == "log":
        x = xr.apply_ufunc(lambda y: np.log1p(9*y)/np.log(10), x, dask="allowed")
    return x

def _to_uint8_img(arr01, size=512):
    arr01 = np.asarray(arr01)
    arr01 = np.clip(arr01, 0, 1)
    if arr01.ndim == 2:
        arr01 = np.stack([arr01, arr01, arr01], axis=-1)
    img = Image.fromarray((arr01 * 255).astype(np.uint8))
    return img.resize((size, size), resample=Image.BILINEAR)

# ---------------- core: precompute once for whole period ----------------
def precompute_daily_channels(ds, mean_msl):
    """
    ds: Dataset with msl,u10,v10,tcwv (time, latitude, longitude), native 0–360 lon okay
    mean_msl: Dataset with climatology on 'doy' (1..366) and same lat/lon
    Returns daily DataArrays (time,lat,lon): R*, G*, B* in 0..1, plus unscaled if needed.
    """
    # --- pull variables
    msl  = _get(ds, ["msl", "MSL", "mslp", "msl_hPa"])
    u10  = _get(ds, ["u10", "VAR_10U"])
    v10  = _get(ds, ["v10", "VAR_10V"])
    tcwv = _get(ds, ["tcwv", "TCWV"])

    # MSL → hPa if needed
    if (msl.attrs.get("units","").lower() in ("pa","pascal","pascals")):
        msl = msl / 100.0
        msl.attrs["units"] = "hPa"

    # ---- daily aggregates (vectorized over the whole time span)
    # wind speed first (keeps lazy)
    ws10 = xr.apply_ufunc(np.hypot, u10, v10, dask="parallelized", output_dtypes=[u10.dtype])

    # resample once
    msl_daymean   = msl.resample(time="1D").mean()
    ws10_daymax   = ws10.resample(time="1D").max()
    tcwv_daymean  = tcwv.resample(time="1D").mean()

    # ---- MSL anomaly vs climatology (broadcast over DOY)
    clim = _get(mean_msl, ["MSL", "msl", "msl_mean", "msl_hPa"])  # expect dims ('doy','latitude','longitude')
    doy_index = msl_daymean["time"].dt.dayofyear
    # pick matching DOY from clim and align grid; drop the 'doy' dim after selection
    clim_on_time = clim.sel(doy=doy_index).drop_vars("doy")
    # ensure same dim order for cheap arithmetic
    msl_daymean  = msl_daymean.transpose("time","latitude","longitude")
    clim_on_time = clim_on_time.transpose("time","latitude","longitude")
    msl_anom = (msl_daymean - clim_on_time)

    # ---- normalize once for all days (still lazy)
    R = 1.0 - _rescale01(msl_anom, *R_RANGE)                  # lows bright
    G = _rescale01(ws10_daymax, *G_RANGE)                    # intensity
    B = _rescale01(tcwv_daymean, *B_RANGE, nonlin=TCWV_NONLIN)  # moist footprint

    # Optional: rechunk for fast per-day slicing (time=1 works well when saving day by day)
    R = R.chunk({"time": 1}); G = G.chunk({"time": 1}); B = B.chunk({"time": 1})

    return R, G, B
    
def _ensure_dirs(root: Path):
    (root/"msl").mkdir(parents=True, exist_ok=True)
    (root/"wmax").mkdir(parents=True, exist_ok=True)
    (root/"tcwv").mkdir(parents=True, exist_ok=True)
    (root/"rgb").mkdir(parents=True, exist_ok=True)

# ---------------- saver: iterate only to write files ----------------
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

def save_range_fast(R, G, B, start, end, out_root=OUT_ROOT, size=SAVE_SIZE, quality=JPEG_QUALITY):
    out_root = Path(out_root)
    _ensure_dirs(out_root)
    dates = pd.date_range(start=start, end=end, freq="D")
    total = len(dates)
    counter = 0
    lock = threading.Lock()

    def _save_one(ts):
        nonlocal counter
        ymd = pd.to_datetime(ts).strftime("%Y%m%d")
        try:
            r = R.sel(time=[ts]).isel(time=0)
            g = G.sel(time=[ts]).isel(time=0)
            b = B.sel(time=[ts]).isel(time=0)
            rgb = xr.concat([r, g, b], dim="channel").transpose("latitude","longitude","channel")

            img_rgb = _to_uint8_img(rgb.values, size=size)
            img_rgb.save(out_root/"rgb"/f"{ymd}_rgb.jpeg", "JPEG", quality=quality, optimize=True)
        except Exception as e:
            print(f"[skip] {ymd}: {e}")
        finally:
            with lock:
                counter += 1
                print(f"\r[{counter}/{total}] processed {ymd}", end="", flush=True)

    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futures = [ex.submit(_save_one, t) for t in dates]
        for f in as_completed(futures):
            f.result()
    print(f"\n✅ Finished writing {total} images to {out_root}/rgb/")


def save_range_fast_single(R, start, end, out_root=OUT_ROOT, size=SAVE_SIZE, quality=JPEG_QUALITY):
    out_root = Path(out_root)
    _ensure_dirs(out_root)
    dates = pd.date_range(start=start, end=end, freq="D")

    def _save_one(ts):
        ymd = pd.to_datetime(ts).strftime("%Y%m%d")
        try:
            r = R.sel(time=[ts]).isel(time=0).values  # ndarray (H,W)
    
            # build (H,W,3), red only
            H, W = r.shape
            rgb = np.zeros((H, W, 3), dtype=np.float32)
            rgb[..., 0] = r  # red channel
    
            img_rgb = _to_uint8_img(rgb, size=size)
            img_rgb.save(out_root/"rgb"/f"{ymd}_rgb.jpeg", "JPEG", quality=quality, optimize=True)
        except Exception as e:
            print(f"[skip] {ymd}: {e}")

    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futures = [ex.submit(_save_one, t) for t in dates]
        for f in futures:
            f.result()



In [8]:
# DOY climatology (Zarr) for MSLP
file_msl = "/glade/work/ncheruku/research/era5_climatology/era5_climatology_1991_2020/MSL_Mean_1991_2020.zarr"
mean_msl = xr.open_dataset(file_msl, engine="zarr")  # expect a var like 'MSL' and coord 'doy'


In [9]:
R, G, B = precompute_daily_channels(ds, mean_msl)

In [10]:
save_range_fast(R, G, B, start="2017-06-01", end="2017-12-01")

[184/184] processed 20171201
✅ Finished writing 184 images to openclip_ready_512/rgb/


In [11]:
# save_range_fast_single(R, start="2017-06-01", end="2017-11-30")